<a href="https://colab.research.google.com/github/ROHAN-BHUTANI/MediTriageAI/blob/main/EPATH_CO_REASON_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E-PATH-CO-REASON: Production-Grade Google Colab Training Notebook

This notebook serves as the **unified, production-grade training and evaluation pipeline** for the E-PATH-CO-REASON research experiments. It is designed to run end-to-end on a completely fresh Google Colab GPU runtime (or a local environment) with a single "Run All" command.

## 1. Notebook Purpose
This notebook orchestrates the training, validation, evaluation, and logging of the **E-PATH-CO-REASON** model. It includes mechanisms for differentiable path routing via Gumbel-Softmax, representation alignment via Dynamic Consistency Projection (DCP), and multi-objective composite loss tracking.

## 2. Directory Layout & Persistence
All outputs are persistently saved inside the experiments workspace folder:
```
experiments/<experiment_name>/
├── best_model.pt             # Best model checkpoint
├── latest_model.pt           # Last completed epoch checkpoint
├── checkpoints/              # Checkpoint directory
├── logs/                     # environment.json and configuration.json
├── exports/                  # training_history.csv, metrics.json, routing_statistics.json
└── figures/                  # loss_curves.png, accuracy_curves.png, confusion matrices
```

## 3. Running & Resuming
- **First Run**: Select `Runtime -> Run All`. The notebook will automatically check imports, setup the repository, validate the dataset, and start training from scratch.
- **Resuming Interrupted Runs**: If training gets disconnected, select `Runtime -> Run All` again. The notebook automatically mounts Google Drive, detects the existing `latest_model.pt` checkpoint, and resumes training from the exact interrupted epoch/optimizer/scheduler/seed state.

## 1. Central Experiment Configuration Block

In [36]:
# ==========================================
# CENTRAL EXPERIMENT CONFIGURATION BLOCK
# ==========================================
EXPERIMENT_CONFIG = {
    "experiment_name": "epath_co_reason_baseline",
    "git_branch": "main",  # Branch to clone/checkout if not already local

    # Dataset Parameters
    "dataset_relative_path": "meditriage/data/processed/dataset.csv",
    "max_samples": None,  # Set to an integer (e.g. 500) to train on subset, or None for the full dataset
    "max_length": 128,

    # Split Parameters
    "train_ratio": 0.8,
    "val_ratio": 0.1,
    "test_ratio": 0.1,

    # Trainer Parameters
    "epochs": 10,
    "batch_size": 32,
    "learning_rate": 1e-4,
    "encoder_lr": 2e-5,
    "weight_decay": 0.01,
    "gradient_clipping": 1.0,
    "gradient_accumulation_steps": 1,
    "use_amp": True,
    "seed": 1337,
    "optimizer_type": "adamw",
    "scheduler_type": "cosine",
    "warmup_ratio": 0.1,

    # Early Stopping
    "early_stopping_patience": 3,
    "early_stopping_metric": "val_loss",
    "early_stopping_min_improvement": 1e-4,

    # Storage and Environment
    "use_drive": True,
    "drive_workspace_dir": "/content/drive/MyDrive/MediTriageAI"
}

## 2. Automatic Repository Setup & Import Validation

In [ ]:
# ==========================================
# SECTION 1 & 3: REPOSITORY DETECT & IMPORT CHECKS
# ==========================================
import os
import sys
from pathlib import Path

# 1. Detect if in Google Colab
is_colab = False
try:
    import google.colab
    is_colab = True
except ImportError:
    pass

if is_colab:
    repo_name = "MediTriageAI"
    expected_repo_root = Path("/content") / repo_name
    if not expected_repo_root.exists():
        REPO_URL = "https://github.com/ROHAN-BHUTANI/MediTriageAI.git"
        print(f"Cloning fresh repository: {REPO_URL} on branch '{EXPERIMENT_CONFIG["git_branch"]}'...")
        !git clone -b {EXPERIMENT_CONFIG["git_branch"]} {REPO_URL} {expected_repo_root}
    repo_root = expected_repo_root.resolve()
else:
    def find_repo_root(start_dir: Path) -> Path:
        for parent in [start_dir] + list(start_dir.parents):
            if (parent / ".git").exists() or (parent / "requirements.txt").exists():
                return parent
        return start_dir
    repo_root = find_repo_root(Path(os.getcwd()).resolve())

os.chdir(repo_root)

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Current Working Directory: {os.getcwd()}")
print(f"Repository Root:           {repo_root}")
print(f"sys.path[0]:               {sys.path[0]}")

req_path = repo_root / "requirements.txt"
if req_path.exists():
    print(f"Installing dependencies from: {req_path}")
    !pip install -q -r {req_path}
else:
    print(f"Warning: requirements.txt not found at {req_path}. Performing fallback install...")
    !pip install -q transformers scikit-learn matplotlib seaborn pandas numpy torch psutil

print("Validating imports and workspace modules...")
required_imports = [
    ("torch", "torch"),
    ("transformers", "transformers"),
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("psutil", "psutil"),
    ("models.emergent_path_triage.model", "E-PATH-CO-REASON Model"),
    ("src.data_pipeline", "E-PATH-CO-REASON Data Pipeline"),
    ("src.trainer", "E-PATH-CO-REASON Trainer")
]

missing = []
for mod_name, friendly_name in required_imports:
    try:
        __import__(mod_name)
        print(f"  ✓ {mod_name} imported successfully.")
    except ImportError as e:
        print(f"  ✗ Failed to import {mod_name}: {e}")
        missing.append(friendly_name)

if missing:
    raise ImportError(f"Verification failed. Missing required components: {missing}")
print("All imports validated successfully.")


## 3. Environment Validation

In [ ]:
# ==========================================
# SECTION 2: ENVIRONMENT VALIDATION & HEALTH REPORT
# ==========================================
import json
import sys
import torch
import psutil
import pandas as pd
import transformers
from src.data_pipeline import detect_colab_environment

env_meta = detect_colab_environment()
has_gpu = env_meta["has_gpu"]
gpu_name = env_meta["gpu_name"]
total_vram = 0
free_vram = 0

if has_gpu:
    t = torch.cuda.get_device_properties(0).total_memory
    a = torch.cuda.memory_allocated(0)
    total_vram = t / (1024 ** 3)
    free_vram = (t - a) / (1024 ** 3)

git_commit = "N/A"
try:
    import subprocess
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(repo_root)).decode("utf-8").strip()
except Exception:
    pass

env_info = {
    "python_version": sys.version,
    "pytorch_version": torch.__version__,
    "cuda_version": torch.version.cuda if has_gpu else "N/A",
    "transformers_version": transformers.__version__,
    "gpu_model": gpu_name,
    "total_gpu_memory_gb": total_vram,
    "available_gpu_memory_gb": free_vram,
    "cpu_cores": psutil.cpu_count(logical=True),
    "ram_gb": psutil.virtual_memory().total / (1024 ** 3),
    "git_commit_hash": git_commit
}

print("ENVIRONMENT AUDIT LOG:")
print(json.dumps(env_info, indent=4))

# PRINT NOTEBOOK HEALTH REPORT
def print_notebook_health_report():
    print("="*60)
    print("E-PATH-CO-REASON NOTEBOOK HEALTH REPORT")
    print("="*60)
    print(f"  Repository Commit    : {env_info['git_commit_hash']}")
    print(f"  PyTorch Version      : {env_info['pytorch_version']}")
    print(f"  CUDA Version         : {env_info['cuda_version']}")
    print(f"  GPU Model            : {env_info['gpu_model']}")
    
    # Dataset rows
    dataset_rows = "N/A"
    config = globals().get("EXPERIMENT_CONFIG", None)
    if config:
        dataset_csv = repo_root / config["dataset_relative_path"]
        if dataset_csv.exists():
            try:
                df = pd.read_csv(dataset_csv)
                dataset_rows = len(df)
            except Exception:
                pass
    print(f"  Dataset Total Rows   : {dataset_rows}")
    
    # Experiment / Checkpoint dirs
    exp_dir = "N/A"
    ckpt_dir = "N/A"
    dirs = globals().get("dirs", None)
    if dirs:
        exp_dir = str(dirs.get("experiments", "N/A"))
        ckpt_dir = str(dirs.get("checkpoints", "N/A"))
    elif config:
        is_colab = False
        try:
            import google.colab
            is_colab = True
        except ImportError:
            pass
        drive_base = Path(config["drive_workspace_dir"]) if config["use_drive"] and is_colab else repo_root
        exp_base = drive_base / "experiments" / config["experiment_name"]
        exp_dir = str(exp_base)
        ckpt_dir = str(exp_base / "checkpoints")
        
    print(f"  Experiment Directory : {exp_dir}")
    print(f"  Checkpoint Directory : {ckpt_dir}")
    print("="*60)

print_notebook_health_report()


## 4. Google Drive Mount & Workspace Setup

In [ ]:
# ==========================================
# SECTION 3: DRIVE MOUNT & FOLDERS VERIFICATION
# ==========================================
from pathlib import Path
import json
import sys
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_AVAILABLE = True
except Exception as e:
    print(f"Drive mount failed: {e}")
    print("Using local /content workspace instead.")
    DRIVE_AVAILABLE = False

# -------------------------
# Workspace selection
# -------------------------
if DRIVE_AVAILABLE:
    drive_base = Path(EXPERIMENT_CONFIG["drive_workspace_dir"])
else:
    # Resolve repo_root dynamically if Cell 10 was skipped
    if 'repo_root' not in globals() and 'repo_root' not in locals():
        def find_repo_root(start_dir: Path) -> Path:
            for parent in [start_dir] + list(start_dir.parents):
                if (parent / ".git").exists() or (parent / "requirements.txt").exists():
                    return parent
            return start_dir
        repo_root = find_repo_root(Path(os.getcwd()).resolve())
    drive_base = repo_root

print(f"Workspace: {drive_base}")

exp_base = drive_base / "experiments" / EXPERIMENT_CONFIG["experiment_name"]

dirs = {
    "experiments": exp_base,
    "checkpoints": exp_base / "checkpoints",
    "logs": exp_base / "logs",
    "figures": exp_base / "figures",
    "exports": exp_base / "exports"
}

for name, path in dirs.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"Verified folder '{name}': {path}")

# Write permission check
test_file = exp_base / "write_check.txt"
try:
    test_file.write_text("write check successful")
    test_file.unlink()
    print("Write permissions successfully verified.")
except Exception as e:
    raise PermissionError(f"Target folder is not writable: {e}")

# Save environment.json with auto-recovery if skipped Cell 12
if 'env_info' in globals() or 'env_info' in locals():
    target_env = env_info
else:
    import torch
    import psutil
    import transformers
    has_gpu = torch.cuda.is_available()
    target_env = {
        "python_version": sys.version,
        "pytorch_version": torch.__version__,
        "cuda_version": torch.version.cuda if has_gpu else "N/A",
        "transformers_version": transformers.__version__,
        "gpu_model": torch.cuda.get_device_name(0) if has_gpu else "N/A",
        "total_gpu_memory_gb": torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if has_gpu else 0,
        "available_gpu_memory_gb": 0,
        "cpu_cores": psutil.cpu_count(logical=True),
        "ram_gb": psutil.virtual_memory().total / (1024 ** 3),
        "git_commit_hash": "N/A"
    }
with open(dirs["logs"] / "environment.json", "w", encoding="utf-8") as f:
    json.dump(target_env, f, indent=4)


## 5. Dataset Validation Checks

In [45]:
# ==========================================
# SECTION 4: DATASET VALIDATION
# ==========================================
import pandas as pd
from src.data_pipeline import LabelValidator
from transformers import AutoTokenizer

print("Discovering dataset relative to repo root...")
dataset_csv = repo_root / EXPERIMENT_CONFIG["dataset_relative_path"]

if not dataset_csv.exists():
    raise FileNotFoundError(f"Dataset CSV not found at: {dataset_csv}")

df = pd.read_csv(dataset_csv)
print(f"Loaded dataset with {len(df)} total rows.")

# Column validations
required_cols = ["text", "department_code", "severity_heuristic"]
for col in required_cols:
    if col not in df.columns:
        raise KeyError(f"Dataset schema validation failed. Column '{col}' is missing.")

# Drop NaNs in text
nan_text = df["text"].isna().sum()
if nan_text > 0:
    print(f"Removing {nan_text} rows with missing text...")
    df = df.dropna(subset=["text"])

# Mappings validations
validator = LabelValidator()
invalid_spec = (~df["department_code"].isin(validator.specialist_classes)).sum()
invalid_sev = (~df["severity_heuristic"].isin(validator.severity_labels)).sum()

if invalid_spec > 0:
    raise ValueError(f"Invalid specialty labels count: {invalid_spec}")
if invalid_sev > 0:
    raise ValueError(f"Invalid severity labels count: {invalid_sev}")

print("Validating tokenizer encoding format...")
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
try:
    tokens = tokenizer.encode(df["text"].iloc[0], truncation=True, max_length=EXPERIMENT_CONFIG["max_length"])
    print(f"Tokenizer validation passed. First text encode length: {len(tokens)}")
except Exception as e:
    raise RuntimeError(f"Tokenizer compatibility check failed: {e}")

print("Dataset validation validation PASSED.")

Discovering dataset relative to repo root...
Loaded dataset with 19996 total rows.
Removing 33 rows with missing text...
Validating tokenizer encoding format...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Tokenizer validation passed. First text encode length: 128
Dataset validation validation PASSED.


# E-PATH-CO-REASON Recovery Execution Guide

This guide details the exact cell execution sequences required for different research tasks on a fresh or resumed Google Colab runtime:

### (A) Fresh Training from Scratch
Run all cells in sequence from top to bottom:
1. **Section 1**: Central Experiment Configuration Block (Cell 3)
2. **Section 2**: Automatic Repository Setup & Import Validation (Cell 10)
3. **Section 3**: Environment Validation & Health Report (Cell 12)
4. **Section 4**: Google Drive Mount & Workspace Setup (Cell 14)
5. **Section 5**: Dataset Validation Checks (Cell 16)
6. **Section 6**: Training Initialization & Fit (Cell 17)
7. **Section 8**: Post-Training Evaluation Exports
8. **Section 9**: Final Experiment Summary Report

### (B) Runtime Reset (Resuming Interrupted Training)
If your runtime gets disconnected during active training, run:
1. **Section 1**: Central Experiment Configuration Block (Cell 3)
2. **Section 2**: Automatic Repository Setup & Import Validation (Cell 10)
3. **Section 4**: Google Drive Mount & Workspace Setup (Cell 14)
4. **Section 6**: Training Initialization & Fit (Cell 17)
   * *Note: The fit cell automatically detects the checkpoint `latest_model.pt` in Drive and resumes from the exact epoch state.*
5. **Section 8 & Section 9** (Evaluation & Summary)

### (C) Evaluation Only (No Retraining)
If training is complete and you only want to evaluate `best_model.pt` on a fresh runtime:
1. **Section 1**: Central Experiment Configuration Block (Cell 3)
2. **Section 2**: Automatic Repository Setup & Import Validation (Cell 10)
3. **Section 4**: Google Drive Mount & Workspace Setup (Cell 14)
4. **Section 8**: Post-Training Evaluation Exports
   * *Note: Section 8 automatically invokes `recover_runtime()` to reconstruct all data structures and loaders from configuration and checkpoints without retraining.*
5. **Section 9**: Final Experiment Summary Report

### (D) Report Regeneration / Plot Re-rendering
If any export file or plot is missing or deleted:
1. **Section 1**: Central Experiment Configuration Block (Cell 3)
2. **Section 2**: Automatic Repository Setup & Import Validation (Cell 10)
3. **Section 4**: Google Drive Mount & Workspace Setup (Cell 14)
4. **Section 8**: Post-Training Evaluation Exports
   * *Note: Section 8 checks for missing exports and regenerates only the missing files dynamically.*


## 6. Training Initialization & Checkpoint Resume Loop

In [ ]:
# ==========================================
# SECTION 6: TRAINING INITIALIZATION & FIT
# ==========================================
import json
import torch
from pathlib import Path
from models.emergent_path_triage.model import EmergentPathTriageConfig, EmergentPathTriageModel
from src.data_pipeline import TokenizerPipeline, EmergentTriageDataset, get_dataloader, get_leakage_safe_splits

def configure_model_freezes(model, checkpoint_path):
    if checkpoint_path and Path(checkpoint_path).exists():
        try:
            ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
            if "optimizer_state_dict" in ckpt and "param_groups" in ckpt["optimizer_state_dict"]:
                num_groups = len(ckpt["optimizer_state_dict"]["param_groups"])
                if num_groups == 1:
                    print("Checkpoint was trained with frozen encoder. Freezing encoder parameters for alignment.")
                    for name, param in model.named_parameters():
                        if "encoder" in name:
                            param.requires_grad = False
                else:
                    print(f"Checkpoint was trained with {num_groups} optimizer groups. Leaving encoder parameters unfrozen.")
        except Exception as e:
            print(f"Warning during model freeze configuration: {e}")

print("Preparing dataloaders...")
if EXPERIMENT_CONFIG["max_samples"] is not None:
    df = df.sample(EXPERIMENT_CONFIG["max_samples"], random_state=EXPERIMENT_CONFIG["seed"])

train_df, val_df, test_df = get_leakage_safe_splits(
    df,
    train_ratio=EXPERIMENT_CONFIG["train_ratio"],
    val_ratio=EXPERIMENT_CONFIG["val_ratio"],
    seed=EXPERIMENT_CONFIG["seed"],
    stratify=False
)

pipeline = TokenizerPipeline(tokenizer, max_length=EXPERIMENT_CONFIG["max_length"])

def create_ds(target_df):
    texts = target_df["text"].tolist()
    spec_ids = [validator.validate_specialist(str(c)) for c in target_df["department_code"]]
    sev_ids = [validator.validate_severity(str(l)) for l in target_df["severity_heuristic"]]
    return EmergentTriageDataset(texts, spec_ids, sev_ids, pipeline)

train_loader = get_dataloader(create_ds(train_df), batch_size=EXPERIMENT_CONFIG["batch_size"], shuffle=True)
val_loader = get_dataloader(create_ds(val_df), batch_size=EXPERIMENT_CONFIG["batch_size"], shuffle=False)
test_loader = get_dataloader(create_ds(test_df), batch_size=EXPERIMENT_CONFIG["batch_size"], shuffle=False)

print("Building model...")
config = EmergentPathTriageConfig(latent_dim=8)
model_meta = EmergentPathTriageModel()
model = model_meta.build(None, triage_config=config)

# Configure model freezes dynamically before trainer creation
latest_ckpt = dirs["checkpoints"] / "latest_model.pt"
checkpoint_to_check = latest_ckpt if latest_ckpt.exists() else None
configure_model_freezes(model, checkpoint_to_check)

from src.trainer import EmergentTrainer, EmergentTrainerConfig
trainer_cfg = EmergentTrainerConfig(
    epochs=EXPERIMENT_CONFIG["epochs"],
    learning_rate=EXPERIMENT_CONFIG["learning_rate"],
    encoder_lr=EXPERIMENT_CONFIG["encoder_lr"],
    weight_decay=EXPERIMENT_CONFIG["weight_decay"],
    gradient_clipping=EXPERIMENT_CONFIG["gradient_clipping"],
    gradient_accumulation_steps=EXPERIMENT_CONFIG["gradient_accumulation_steps"],
    use_amp=EXPERIMENT_CONFIG["use_amp"],
    early_stopping_patience=EXPERIMENT_CONFIG["early_stopping_patience"],
    early_stopping_metric=EXPERIMENT_CONFIG["early_stopping_metric"],
    early_stopping_min_improvement=EXPERIMENT_CONFIG["early_stopping_min_improvement"],
    warmup_ratio=EXPERIMENT_CONFIG["warmup_ratio"],
    seed=EXPERIMENT_CONFIG["seed"],
    optimizer_type=EXPERIMENT_CONFIG["optimizer_type"],
    scheduler_type=EXPERIMENT_CONFIG["scheduler_type"],
    checkpoint_dir=str(dirs["checkpoints"])
)

trainer = EmergentTrainer(
    model=model,
    config=trainer_cfg,
    train_loader=train_loader,
    val_loader=val_loader,
    tokenizer=tokenizer
)
trainer.checkpoint_dir = dirs["checkpoints"]

# Checkpoint Resume loop check
if latest_ckpt.exists():
    print(f"==================================================")
    print(f"Checkpoint discovered: {latest_ckpt}. Resuming run...")
    print(f"==================================================")
    trainer.load_checkpoint(latest_ckpt)
else:
    print(f"==================================================")
    print(f"No checkpoint found under checkpoints/. Initiating new training.")
    print(f"==================================================")

best_val_metrics = trainer.fit()

# E-PATH-CO-REASON Recovery Execution Guide

This guide details the exact cell execution sequences required for different research tasks on a fresh or resumed Google Colab runtime:

### (A) Fresh Training from Scratch
Run all cells in sequence from top to bottom:
1. **Section 1**: Central Experiment Configuration Block (Cell 3)
2. **Section 2**: Automatic Repository Setup & Import Validation (Cell 10)
3. **Section 3**: Environment Validation & Health Report (Cell 12)
4. **Section 4**: Google Drive Mount & Workspace Setup (Cell 14)
5. **Section 5**: Dataset Validation Checks (Cell 16)
6. **Section 6**: Training Initialization & Fit (Cell 17)
7. **Section 8**: Post-Training Evaluation Exports
8. **Section 9**: Final Experiment Summary Report

### (B) Runtime Reset (Resuming Interrupted Training)
If your runtime gets disconnected during active training, run:
1. **Section 1**: Central Experiment Configuration Block (Cell 3)
2. **Section 2**: Automatic Repository Setup & Import Validation (Cell 10)
3. **Section 4**: Google Drive Mount & Workspace Setup (Cell 14)
4. **Section 6**: Training Initialization & Fit (Cell 17)
   * *Note: The fit cell automatically detects the checkpoint `latest_model.pt` in Drive and resumes from the exact epoch state.*
5. **Section 8 & Section 9** (Evaluation & Summary)

### (C) Evaluation Only (No Retraining)
If training is complete and you only want to evaluate `best_model.pt` on a fresh runtime:
1. **Section 1**: Central Experiment Configuration Block (Cell 3)
2. **Section 2**: Automatic Repository Setup & Import Validation (Cell 10)
3. **Section 4**: Google Drive Mount & Workspace Setup (Cell 14)
4. **Section 8**: Post-Training Evaluation Exports
   * *Note: Section 8 automatically invokes `recover_runtime()` to reconstruct all data structures and loaders from configuration and checkpoints without retraining.*
5. **Section 9**: Final Experiment Summary Report

### (D) Report Regeneration / Plot Re-rendering
If any export file or plot is missing or deleted:
1. **Section 1**: Central Experiment Configuration Block (Cell 3)
2. **Section 2**: Automatic Repository Setup & Import Validation (Cell 10)
3. **Section 4**: Google Drive Mount & Workspace Setup (Cell 14)
4. **Section 8**: Post-Training Evaluation Exports
   * *Note: Section 8 checks for missing exports and regenerates only the missing files dynamically.*


In [ ]:
# ==========================================
# SECTION 8: EVALUATION METRICS & CONFUSION PLOTS (WITH RUNTIME RECOVERY)
# ==========================================
import os
import sys
import json
import shutil
from pathlib import Path
import datetime
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score
from transformers import AutoTokenizer
from models.emergent_path_triage.model import EmergentPathTriageConfig, EmergentPathTriageModel
from src.data_pipeline import TokenizerPipeline, EmergentTriageDataset, get_dataloader, get_leakage_safe_splits, LabelValidator
from src.trainer import EmergentTrainer, EmergentTrainerConfig

def find_best_checkpoint(drive_workspace_dir, repo_root):
    is_colab = False
    try:
        import google.colab
        is_colab = True
    except ImportError:
        pass
    drive_base = Path(drive_workspace_dir) if is_colab else repo_root
    exp_dir = drive_base / "experiments"
    checkpoints = []
    if exp_dir.exists():
        for exp_folder in exp_dir.iterdir():
            if exp_folder.is_dir():
                ckpt_file = exp_folder / "checkpoints" / "best_model.pt"
                if ckpt_file.exists():
                    mtime = ckpt_file.stat().st_mtime
                    checkpoints.append((exp_folder.name, ckpt_file, mtime))
    if not checkpoints:
        local_exp_dir = repo_root / "experiments"
        if local_exp_dir.exists():
            for exp_folder in local_exp_dir.iterdir():
                if exp_folder.is_dir():
                    ckpt_file = exp_folder / "checkpoints" / "best_model.pt"
                    if ckpt_file.exists():
                        mtime = ckpt_file.stat().st_mtime
                        checkpoints.append((exp_folder.name, ckpt_file, mtime))
    if not checkpoints:
        raise FileNotFoundError("No 'best_model.pt' checkpoint discovered under experiments/**/checkpoints/")
    checkpoints.sort(key=lambda x: x[2], reverse=True)
    return checkpoints[0]

def validate_checkpoint(ckpt_path):
    try:
        checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    except Exception as e:
        raise RuntimeError(f"Checkpoint Loading Failed: The file at {ckpt_path} is corrupt or not a valid PyTorch save file. Error: {e}")
    required_keys = ["model_state_dict", "optimizer_state_dict", "history", "metadata"]
    missing_keys = [k for k in required_keys if k not in checkpoint]
    if missing_keys:
        raise KeyError(f"Checkpoint Validation Failed: The checkpoint at {ckpt_path} is missing critical components: {missing_keys}.")
    print("Checkpoint contents successfully validated.")
    return checkpoint

def recover_runtime(experiment_name=None):
    """
    Reconstructs all variables required for E-PATH-CO-REASON evaluation
    after a session reset or runtime restart.
    """
    # Redefine helper functions inside recover_runtime scoping
    def _find_best_checkpoint(drive_workspace_dir, repo_root):
        is_colab = False
        try:
            import google.colab
            is_colab = True
        except ImportError:
            pass
        drive_base = Path(drive_workspace_dir) if is_colab else repo_root
        exp_dir = drive_base / "experiments"
        checkpoints = []
        if exp_dir.exists():
            for exp_folder in exp_dir.iterdir():
                if exp_folder.is_dir():
                    ckpt_file = exp_folder / "checkpoints" / "best_model.pt"
                    if ckpt_file.exists():
                        mtime = ckpt_file.stat().st_mtime
                        checkpoints.append((exp_folder.name, ckpt_file, mtime))
        if not checkpoints:
            local_exp_dir = repo_root / "experiments"
            if local_exp_dir.exists():
                for exp_folder in local_exp_dir.iterdir():
                    if exp_folder.is_dir():
                        ckpt_file = exp_folder / "checkpoints" / "best_model.pt"
                        if ckpt_file.exists():
                            mtime = ckpt_file.stat().st_mtime
                            checkpoints.append((exp_folder.name, ckpt_file, mtime))
        if not checkpoints:
            raise FileNotFoundError("No 'best_model.pt' checkpoint discovered under experiments/**/checkpoints/")
        checkpoints.sort(key=lambda x: x[2], reverse=True)
        return checkpoints[0]

    def _configure_model_freezes(model, checkpoint_path):
        if checkpoint_path and Path(checkpoint_path).exists():
            try:
                ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
                if "optimizer_state_dict" in ckpt and "param_groups" in ckpt["optimizer_state_dict"]:
                    num_groups = len(ckpt["optimizer_state_dict"]["param_groups"])
                    if num_groups == 1:
                        print("Checkpoint was trained with frozen encoder. Freezing encoder parameters for alignment.")
                        for name, param in model.named_parameters():
                            if "encoder" in name:
                                param.requires_grad = False
                    else:
                        print(f"Checkpoint was trained with {num_groups} optimizer groups. Leaving encoder parameters unfrozen.")
            except Exception as e:
                print(f"Warning during model freeze configuration: {e}")

    print("Reconstructing runtime environments...")
    def find_repo_root(start_dir: Path) -> Path:
        for parent in [start_dir] + list(start_dir.parents):
            if (parent / ".git").exists() or (parent / "requirements.txt").exists():
                return parent
        return start_dir
    repo_root = find_repo_root(Path(os.getcwd()).resolve())
    os.chdir(repo_root)
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
        
    g_config = globals().get("EXPERIMENT_CONFIG", None)
    drive_workspace = g_config["drive_workspace_dir"] if g_config else "/content/drive/MyDrive/MediTriageAI"
    
    # Checkpoint Discovery
    try:
        exp_name_found, best_ckpt_found, mtime_found = _find_best_checkpoint(drive_workspace, repo_root)
        experiment_name = exp_name_found
        best_ckpt_path = best_ckpt_found
        timestamp = datetime.datetime.fromtimestamp(mtime_found).strftime('%Y-%m-%d %H:%M:%S')
        print("="*50)
        print(f"Experiment selected : {experiment_name}")
        print(f"Checkpoint path     : {best_ckpt_path}")
        print(f"Training timestamp  : {timestamp}")
        print("="*50)
    except FileNotFoundError:
        experiment_name = g_config["experiment_name"] if g_config else "epath_co_reason_baseline"
        best_ckpt_path = None
        
    # Drive configuration
    is_colab = False
    try:
        import google.colab
        is_colab = True
    except ImportError:
        pass
        
    drive_base = Path(drive_workspace) if is_colab else repo_root
    exp_base = drive_base / "experiments" / experiment_name
    
    # Load configuration.json if available
    config_json = exp_base / "logs" / "configuration.json"
    if config_json.exists():
        with open(config_json, "r") as f:
            config = json.load(f)
    else:
        config = g_config if g_config else {
            "experiment_name": experiment_name,
            "git_branch": "main",
            "dataset_relative_path": "meditriage/data/processed/dataset.csv",
            "max_samples": None,
            "max_length": 128,
            "train_ratio": 0.8,
            "val_ratio": 0.1,
            "test_ratio": 0.1,
            "epochs": 10,
            "batch_size": 32,
            "learning_rate": 1e-4,
            "encoder_lr": 2e-5,
            "weight_decay": 0.01,
            "gradient_clipping": 1.0,
            "gradient_accumulation_steps": 1,
            "use_amp": True,
            "seed": 1337,
            "optimizer_type": "adamw",
            "scheduler_type": "cosine",
            "warmup_ratio": 0.1,
            "early_stopping_patience": 3,
            "early_stopping_metric": "val_loss",
            "early_stopping_min_improvement": 1e-4,
            "use_drive": True,
            "drive_workspace_dir": str(drive_base)
        }
        
    dirs = {
        "experiments": exp_base,
        "checkpoints": exp_base / "checkpoints",
        "logs": exp_base / "logs",
        "figures": exp_base / "figures",
        "exports": exp_base / "exports"
    }
    
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
        
    tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
    validator = LabelValidator()
    
    dataset_csv = repo_root / config["dataset_relative_path"]
    df = pd.read_csv(dataset_csv)
    if df["text"].isna().sum() > 0:
        df = df.dropna(subset=["text"])
    if config["max_samples"] is not None:
        df = df.sample(config["max_samples"], random_state=config["seed"])
        
    train_df, val_df, test_df = get_leakage_safe_splits(
        df,
        train_ratio=config["train_ratio"],
        val_ratio=config["val_ratio"],
        seed=config["seed"],
        stratify=False
    )
    pipeline = TokenizerPipeline(tokenizer, max_length=config["max_length"])
    
    def create_ds(target_df):
        texts = target_df["text"].tolist()
        spec_ids = [validator.validate_specialist(str(c)) for c in target_df["department_code"]]
        sev_ids = [validator.validate_severity(str(l)) for l in target_df["severity_heuristic"]]
        return EmergentTriageDataset(texts, spec_ids, sev_ids, pipeline)

    train_loader = get_dataloader(create_ds(train_df), batch_size=config["batch_size"], shuffle=True)
    val_loader = get_dataloader(create_ds(val_df), batch_size=config["batch_size"], shuffle=False)
    test_loader = get_dataloader(create_ds(test_df), batch_size=config["batch_size"], shuffle=False)
    
    triage_config = EmergentPathTriageConfig(latent_dim=8)
    model_meta = EmergentPathTriageModel()
    model = model_meta.build(None, triage_config=triage_config)
    
    # Align model parameters dynamically based on best checkpoint groups count
    _configure_model_freezes(model, best_ckpt_path)
    
    trainer_cfg = EmergentTrainerConfig(
        epochs=config["epochs"],
        learning_rate=config["learning_rate"],
        encoder_lr=config["encoder_lr"],
        weight_decay=config["weight_decay"],
        gradient_clipping=config["gradient_clipping"],
        gradient_accumulation_steps=config["gradient_accumulation_steps"],
        use_amp=config["use_amp"],
        early_stopping_patience=config["early_stopping_patience"],
        early_stopping_metric=config["early_stopping_metric"],
        early_stopping_min_improvement=config["early_stopping_min_improvement"],
        warmup_ratio=config["warmup_ratio"],
        seed=config["seed"],
        optimizer_type=config["optimizer_type"],
        scheduler_type=config["scheduler_type"],
        checkpoint_dir=str(dirs["checkpoints"])
    )
    
    trainer = EmergentTrainer(
        model=model,
        config=trainer_cfg,
        train_loader=train_loader,
        val_loader=val_loader,
        tokenizer=tokenizer
    )
    trainer.checkpoint_dir = dirs["checkpoints"]
    
    def fit_override(*args, **kwargs):
        raise RuntimeError("CRITICAL ERROR: trainer.fit() was invoked during the evaluation flow! Training is strictly disabled during evaluation.")
    trainer.fit = fit_override
    
    return config, dirs, tokenizer, validator, train_loader, val_loader, test_loader, model, trainer

# Perform Runtime Recovery
config, dirs, tokenizer, validator, train_loader, val_loader, test_loader, model, trainer = recover_runtime()

# Resolve Best Checkpoint Path
g_config = globals().get("EXPERIMENT_CONFIG", None)
drive_workspace = g_config["drive_workspace_dir"] if g_config else "/content/drive/MyDrive/MediTriageAI"
def find_best_checkpoint_path():
    try:
        _, ckpt_path, _ = find_best_checkpoint(drive_workspace, Path(os.getcwd()).resolve())
        return ckpt_path
    except FileNotFoundError:
        return dirs["checkpoints"] / "best_model.pt"

best_ckpt = find_best_checkpoint_path()

# Self-Validation
def perform_self_validation(tokenizer, validator, model, trainer, checkpoint_path, test_loader):
    checks = {
        "Tokenizer instantiated": tokenizer is not None,
        "Label Validator instantiated": validator is not None,
        "Model instantiated": model is not None,
        "Trainer instantiated": trainer is not None,
        "Checkpoint file exists": Path(checkpoint_path).exists() if checkpoint_path else False,
        "Checkpoint readable": False,
        "Test loader not empty": len(test_loader) > 0 if test_loader is not None else False
    }
    if checks["Checkpoint file exists"]:
        try:
            validate_checkpoint(checkpoint_path)
            checks["Checkpoint readable"] = True
        except Exception as e:
            print(f"Checkpoint validation check failed: {e}")
            checks["Checkpoint readable"] = False
            
    print("\n" + "="*50)
    print("PRE-EVALUATION SELF-VALIDATION CHECKLIST:")
    print("="*50)
    all_ok = True
    for desc, passed in checks.items():
        status = "✓ PASSED" if passed else "✗ FAILED"
        print(f"  {status:<10} | {desc}")
        if not passed:
            all_ok = False
    print("="*50)
    
    if not all_ok:
        critical_failed = not (checks["Tokenizer instantiated"] and checks["Label Validator instantiated"] and checks["Model instantiated"] and checks["Trainer instantiated"] and checks["Checkpoint readable"] and checks["Test loader not empty"])
        if critical_failed:
            raise RuntimeError("CRITICAL SELF-VALIDATION FAILURE: One or more critical requirements failed. Aborting evaluation flow.")
    print("All critical self-validation checks passed successfully!")

perform_self_validation(tokenizer, validator, model, trainer, best_ckpt, test_loader)

# Load best parameters checkpoint
print("Loading best parameters checkpoint...")
trainer.load_checkpoint(best_ckpt)

# Dynamically resolve best_val_metrics if it is missing
if 'best_val_metrics' not in globals() or best_val_metrics is None:
    if hasattr(trainer, 'history') and trainer.history:
        best_val_metrics = min(trainer.history, key=lambda x: x.get("val_loss", float('inf')))
    else:
        best_val_metrics = {
            "val_loss": 0.0,
            "val_specialist_loss": 0.0,
            "val_severity_loss": 0.0,
            "val_cons_loss": 0.0,
            "val_div_loss": 0.0,
            "val_ortho_loss": 0.0,
            "epoch": "N/A"
        }

# Execute Evaluation Pass
model.eval()
all_spec_labels = []
all_sev_labels = []
all_spec_preds = []
all_sev_preds = []
all_routing_probs = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(trainer.device)
        attention_mask = batch["attention_mask"].to(trainer.device)
        labels_spec = batch["labels_specialist"]
        labels_sev = batch["labels_severity"]

        outputs = model(input_ids, attention_mask)

        spec_preds = outputs.specialist_logits.argmax(dim=-1).cpu().numpy()
        sev_preds = outputs.severity_logits.argmax(dim=-1).cpu().numpy()

        all_spec_labels.extend(labels_spec.numpy())
        all_sev_labels.extend(labels_sev.numpy())
        all_spec_preds.extend(spec_preds)
        all_sev_preds.extend(sev_preds)

        if model._last_routing_decision is not None:
            all_routing_probs.append(model._last_routing_decision.routing_probabilities.cpu().numpy())

spec_acc = accuracy_score(all_spec_labels, all_spec_preds)
spec_p, spec_r, spec_f1, _ = precision_recall_fscore_support(all_spec_labels, all_spec_preds, average="macro", zero_division=0)
sev_acc = accuracy_score(all_sev_labels, all_sev_preds)
sev_p, sev_r, sev_f1, _ = precision_recall_fscore_support(all_sev_labels, all_sev_preds, average="macro", zero_division=0)

# Check and regenerate all expected outputs
def verify_and_regenerate_exports(dirs, trainer, validator, model, test_loader, spec_acc, spec_p, spec_r, spec_f1, sev_acc, sev_p, sev_r, sev_f1, best_val_metrics, all_spec_labels, all_spec_preds, all_sev_labels, all_sev_preds, all_routing_probs):
    exports_dir = dirs["exports"]
    figures_dir = dirs["figures"]
    
    required_files = {
        "metrics.json": exports_dir / "metrics.json",
        "routing_statistics.json": exports_dir / "routing_statistics.json",
        "training_history.csv": exports_dir / "training_history.csv",
        "validation_history.csv": exports_dir / "validation_history.csv",
        "loss_curves.png": figures_dir / "loss_curves.png",
        "accuracy_curves.png": figures_dir / "accuracy_curves.png",
        "specialist_confusion_matrix.png": figures_dir / "specialist_confusion_matrix.png",
        "severity_confusion_matrix.png": figures_dir / "severity_confusion_matrix.png"
    }
    
    missing = [name for name, path in required_files.items() if not path.exists()]
    if not missing:
        print("All expected export files verified successfully.")
        return
        
    print(f"Missing exports detected: {missing}. Regenerating missing files...")
    history = getattr(trainer, 'history', [])
    history_df = pd.DataFrame(history)
    
    for name in missing:
        if name == "metrics.json":
            metrics_export = {
                "specialist": {
                    "accuracy": float(spec_acc),
                    "macro_precision": float(spec_p),
                    "macro_recall": float(spec_r),
                    "macro_f1": float(spec_f1)
                },
                "severity": {
                    "accuracy": float(sev_acc),
                    "macro_precision": float(sev_p),
                    "macro_recall": float(sev_r),
                    "macro_f1": float(sev_f1)
                },
                "overall_losses": {
                    "val_loss": float(best_val_metrics["val_loss"]),
                    "val_specialist_loss": float(best_val_metrics["val_specialist_loss"]),
                    "val_severity_loss": float(best_val_metrics["val_severity_loss"]),
                    "val_cons_loss": float(best_val_metrics["val_cons_loss"]),
                    "val_div_loss": float(best_val_metrics["val_div_loss"]),
                    "val_ortho_loss": float(best_val_metrics["val_ortho_loss"])
                }
            }
            with open(required_files["metrics.json"], "w") as f:
                json.dump(metrics_export, f, indent=4)
            print("Regenerated metrics.json")
            
        elif name == "routing_statistics.json":
            if all_routing_probs:
                all_probs = np.concatenate(all_routing_probs, axis=0)
                B_t, M_t, N_t = all_probs.shape
                epsilon = 1e-9
                entropies = -np.sum(all_probs * np.log(all_probs + epsilon), axis=-1)
                util_argmax = all_probs.argmax(axis=-1)
                utilization_counts = [np.bincount(util_argmax[:, s], minlength=N_t).tolist() for s in range(M_t)]
                routing_export = {
                    "mean_routing_entropy": float(entropies.mean()),
                    "entropy_per_step": entropies.mean(axis=0).tolist(),
                    "ctb_utilizations_per_step": utilization_counts,
                    "average_reasoning_depth": M_t,
                    "mean_confidence": float(np.max(all_probs, axis=-1).mean())
                }
                with open(required_files["routing_statistics.json"], "w") as f:
                    json.dump(routing_export, f, indent=4)
                print("Regenerated routing_statistics.json")
                
        elif name == "training_history.csv":
            if not history_df.empty:
                history_df.to_csv(required_files["training_history.csv"], index=False)
                print("Regenerated training_history.csv")
                
        elif name == "validation_history.csv":
            if not history_df.empty:
                val_cols = [c for c in history_df.columns if "val_" in c or c in ["epoch", "time"]]
                history_df[val_cols].to_csv(required_files["validation_history.csv"], index=False)
                print("Regenerated validation_history.csv")
                
        elif name == "loss_curves.png":
            if not history_df.empty:
                plt.figure(figsize=(10, 5))
                plt.plot(history_df["epoch"], history_df["train_loss"], label="Train Loss", marker="o")
                plt.plot(history_df["epoch"], history_df["val_loss"], label="Val Loss", marker="x")
                plt.title("E-PATH-CO-REASON Loss Curves")
                plt.xlabel("Epoch")
                plt.ylabel("Loss")
                plt.grid(True)
                plt.legend()
                plt.savefig(required_files["loss_curves.png"])
                plt.close()
                print("Regenerated loss_curves.png")
                
        elif name == "accuracy_curves.png":
            if not history_df.empty:
                plt.figure(figsize=(10, 5))
                plt.plot(history_df["epoch"], history_df["train_specialist_acc"], label="Train Spec Acc", marker="o")
                plt.plot(history_df["epoch"], history_df["val_specialist_acc"], label="Val Spec Acc", marker="x")
                plt.plot(history_df["epoch"], history_df["train_severity_acc"], label="Train Sev Acc", marker="s")
                plt.plot(history_df["epoch"], history_df["val_severity_acc"], label="Val Sev Acc", marker="d")
                plt.title("E-PATH-CO-REASON Accuracy Curves")
                plt.xlabel("Epoch")
                plt.ylabel("Accuracy")
                plt.grid(True)
                plt.legend()
                plt.savefig(required_files["accuracy_curves.png"])
                plt.close()
                print("Regenerated accuracy_curves.png")
                
        elif name == "specialist_confusion_matrix.png":
            plt.figure(figsize=(10, 8))
            sns.heatmap(confusion_matrix(all_spec_labels, all_spec_preds), annot=True, fmt="d", cmap="Blues",
                        xticklabels=validator.specialist_classes, yticklabels=validator.specialist_classes)
            plt.title("Specialist Confusion Matrix")
            plt.savefig(required_files["specialist_confusion_matrix.png"])
            plt.close()
            print("Regenerated specialist_confusion_matrix.png")
            
        elif name == "severity_confusion_matrix.png":
            plt.figure(figsize=(8, 6))
            sns.heatmap(confusion_matrix(all_sev_labels, all_sev_preds), annot=True, fmt="d", cmap="Oranges",
                        xticklabels=validator.severity_labels, yticklabels=validator.severity_labels)
            plt.title("Severity Confusion Matrix")
            plt.savefig(required_files["severity_confusion_matrix.png"])
            plt.close()
            print("Regenerated severity_confusion_matrix.png")

verify_and_regenerate_exports(dirs, trainer, validator, model, test_loader, spec_acc, spec_p, spec_r, spec_f1, sev_acc, sev_p, sev_r, sev_f1, best_val_metrics, all_spec_labels, all_spec_preds, all_sev_labels, all_sev_preds, all_routing_probs)

shutil.copyfile(dirs["checkpoints"] / "best_model.pt", dirs["experiments"] / "best_model.pt")
shutil.copyfile(dirs["checkpoints"] / "latest_model.pt", dirs["experiments"] / "latest_model.pt")

print("Post-training assets successfully compiled, verified, and exported.")

# E-PATH-CO-REASON Recovery Execution Guide

This guide details the exact cell execution sequences required for different research tasks on a fresh or resumed Google Colab runtime:

### (A) Fresh Training from Scratch
Run all cells in sequence from top to bottom:
1. **Section 1**: Central Experiment Configuration Block (Cell 3)
2. **Section 2**: Automatic Repository Setup & Import Validation (Cell 10)
3. **Section 3**: Environment Validation & Health Report (Cell 12)
4. **Section 4**: Google Drive Mount & Workspace Setup (Cell 14)
5. **Section 5**: Dataset Validation Checks (Cell 16)
6. **Section 6**: Training Initialization & Fit (Cell 17)
7. **Section 8**: Post-Training Evaluation Exports
8. **Section 9**: Final Experiment Summary Report

### (B) Runtime Reset (Resuming Interrupted Training)
If your runtime gets disconnected during active training, run:
1. **Section 1**: Central Experiment Configuration Block (Cell 3)
2. **Section 2**: Automatic Repository Setup & Import Validation (Cell 10)
3. **Section 4**: Google Drive Mount & Workspace Setup (Cell 14)
4. **Section 6**: Training Initialization & Fit (Cell 17)
   * *Note: The fit cell automatically detects the checkpoint `latest_model.pt` in Drive and resumes from the exact epoch state.*
5. **Section 8 & Section 9** (Evaluation & Summary)

### (C) Evaluation Only (No Retraining)
If training is complete and you only want to evaluate `best_model.pt` on a fresh runtime:
1. **Section 1**: Central Experiment Configuration Block (Cell 3)
2. **Section 2**: Automatic Repository Setup & Import Validation (Cell 10)
3. **Section 4**: Google Drive Mount & Workspace Setup (Cell 14)
4. **Section 8**: Post-Training Evaluation Exports
   * *Note: Section 8 automatically invokes `recover_runtime()` to reconstruct all data structures and loaders from configuration and checkpoints without retraining.*
5. **Section 9**: Final Experiment Summary Report

### (D) Report Regeneration / Plot Re-rendering
If any export file or plot is missing or deleted:
1. **Section 1**: Central Experiment Configuration Block (Cell 3)
2. **Section 2**: Automatic Repository Setup & Import Validation (Cell 10)
3. **Section 4**: Google Drive Mount & Workspace Setup (Cell 14)
4. **Section 8**: Post-Training Evaluation Exports
   * *Note: Section 8 checks for missing exports and regenerates only the missing files dynamically.*


In [ ]:
# ==========================================
# SECTION 9: FINAL EXPERIMENT SUMMARY
# ==========================================
if 'dirs' not in globals() or 'EXPERIMENT_CONFIG' not in globals() or 'spec_acc' not in globals():
    config, dirs, tokenizer, validator, train_loader, val_loader, test_loader, model, trainer = recover_runtime()
    if hasattr(trainer, 'history') and trainer.history:
        best_val_metrics = min(trainer.history, key=lambda x: x.get("val_loss", float('inf')))
    else:
        best_val_metrics = {"epoch": "N/A"}
    metrics_json = dirs["exports"] / "metrics.json"
    if metrics_json.exists():
        with open(metrics_json, "r") as f:
            m_exp = json.load(f)
        spec_acc = m_exp["specialist"]["accuracy"]
        sev_acc = m_exp["severity"]["accuracy"]
        spec_f1 = m_exp["specialist"]["macro_f1"]
        sev_f1 = m_exp["severity"]["macro_f1"]
    else:
        spec_acc, sev_acc, spec_f1, sev_f1 = 0.0, 0.0, 0.0, 0.0
else:
    config = EXPERIMENT_CONFIG
        
if 'env_info' not in globals():
    env_json = dirs["logs"] / "environment.json"
    if env_json.exists():
        with open(env_json, "r") as f:
            env_info = json.load(f)
    else:
        env_info = {"gpu_model": "Unknown (Session Reset)"}

print(f"==================================================")
print(f"FINAL EXPERIMENT SUMMARY REPORT")
print(f"==================================================")
print(f"Experiment Name      : {config['experiment_name']}")
print(f"GPU Model Used       : {env_info['gpu_model']}")
print(f"Best Training Epoch  : {best_val_metrics.get('epoch', 'N/A')}")
print(f"Specialist Accuracy  : {spec_acc:.2%}")
print(f"Severity Accuracy    : {sev_acc:.2%}")
print(f"Specialist F1-Score  : {spec_f1:.2%}")
print(f"Severity F1-Score    : {sev_f1:.2%}")
print(f"--------------------------------------------------")
print(f"Outputs Directory Locations:")
print(f"- Checkpoints        : {dirs['checkpoints']}")
print(f"- Exported Reports   : {dirs['exports']}")
print(f"- Plot Figures       : {dirs['figures']}")
print(f"==================================================")

## 10. Diagnostics, Utilities, & Custom Colab Checks

This section aggregates interactive checks, path mappings, and repository structures verified during training.

### 10.1. Full Dtype Trace & Dtype Contract Validation

In [ ]:
# ==========================================
# SECTION 10.1: FULL DTYPE TRACE & CONTRACT VALIDATION
# ==========================================
import json
import torch
from pathlib import Path
from src.model import JointLoss

def run_dtype_trace_and_validate():
    print("="*60)
    print("RUNNING FULL DTYPE TRACE & CONTRACT VALIDATION")
    print("="*60)
    
    # 1. Recover model and trainer
    if 'model' not in globals() or 'trainer' not in globals() or 'test_loader' not in globals():
        print("Reconstructing runtime components...")
        config, dirs, tokenizer, validator, train_loader, val_loader, test_loader, model, trainer = recover_runtime()
    else:
        model = globals()['model']
        trainer = globals()['trainer']
        test_loader = globals()['test_loader']
        dirs = globals()['dirs']
        
    # Get a batch
    batch = next(iter(test_loader))
    
    # Trace store
    trace = {}
    hooks = []
    
    def make_hook(name):
        def hook(module, input, output):
            in_dtypes = []
            if name == "Encoder":
                if len(input) > 0 and isinstance(input[0], torch.Tensor):
                    in_dtypes.append(str(input[0].dtype))
            elif name == "Evidence Synthesizer":
                if len(input) > 0 and isinstance(input[0], torch.Tensor):
                    in_dtypes.append(str(input[0].dtype))
            elif name == "Router":
                evidence = input[0]
                if hasattr(evidence, "symptom"):
                    in_dtypes.append(str(evidence.symptom.dtype))
            elif "Thought Block" in name:
                if len(input) > 0 and isinstance(input[0], torch.Tensor):
                    in_dtypes.append(str(input[0].dtype))
            elif name == "Execution Engine":
                ev_list = input[0]
                if isinstance(ev_list, list):
                    for x in ev_list:
                        if isinstance(x, torch.Tensor):
                            in_dtypes.append(str(x.dtype))
            elif name in ["Consistency Projection", "Prediction Heads (Specialist)", "Prediction Heads (Severity)"]:
                if len(input) > 0 and isinstance(input[0], torch.Tensor):
                    in_dtypes.append(str(input[0].dtype))
            else:
                if isinstance(input, tuple):
                    for x in input:
                        if isinstance(x, torch.Tensor):
                            in_dtypes.append(str(x.dtype))
                elif isinstance(input, torch.Tensor):
                    in_dtypes.append(str(input.dtype))
                    
            out_dtypes = []
            if name == "Encoder":
                if hasattr(output, "last_hidden_state"):
                    out_dtypes.append(str(output.last_hidden_state.dtype))
            elif name == "Evidence Synthesizer":
                if hasattr(output, "symptom"):
                    out_dtypes.append(str(output.symptom.dtype))
            elif name == "Router":
                if hasattr(output, "routing_probabilities"):
                    out_dtypes.append(str(output.routing_probabilities.dtype))
            elif "Thought Block" in name:
                if isinstance(output, torch.Tensor):
                    out_dtypes.append(str(output.dtype))
                elif isinstance(output, tuple) and len(output) > 0 and isinstance(output[0], torch.Tensor):
                    out_dtypes.append(str(output[0].dtype))
            elif name == "Execution Engine":
                if isinstance(output, tuple) and len(output) > 0 and isinstance(output[0], torch.Tensor):
                    out_dtypes.append(str(output[0].dtype))
            elif name == "Consistency Projection":
                if isinstance(output, tuple):
                    for x in output:
                        if isinstance(x, torch.Tensor):
                            out_dtypes.append(str(x.dtype))
            elif name in ["Prediction Heads (Specialist)", "Prediction Heads (Severity)"]:
                if isinstance(output, torch.Tensor):
                    out_dtypes.append(str(output.dtype))
            else:
                if isinstance(output, tuple):
                    for x in output:
                        if isinstance(x, torch.Tensor):
                            out_dtypes.append(str(x.dtype))
                elif isinstance(output, torch.Tensor):
                    out_dtypes.append(str(output.dtype))
                    
            trace[name] = {
                "input_dtypes": in_dtypes,
                "output_dtypes": out_dtypes
            }
        return hook

    # Register hooks
    hooks.append(model.encoder.register_forward_hook(make_hook("Encoder")))
    hooks.append(model.dces.register_forward_hook(make_hook("Evidence Synthesizer")))
    hooks.append(model.router.register_forward_hook(make_hook("Router")))
    for idx, block in enumerate(model.blocks):
        hooks.append(block.register_forward_hook(make_hook(f"Thought Block {idx + 1}")))
    hooks.append(model.engine.register_forward_hook(make_hook("Execution Engine")))
    hooks.append(model.dcp.register_forward_hook(make_hook("Consistency Projection")))
    hooks.append(model.classifier_specialist.register_forward_hook(make_hook("Prediction Heads (Specialist)")))
    hooks.append(model.classifier_severity.register_forward_hook(make_hook("Prediction Heads (Severity)")))
    hooks.append(model.register_forward_hook(make_hook("Model Top Level")))

    # Run forward pass under autocast matching trainer AMP state
    model.eval()
    device_type = "cuda" if next(model.parameters()).device.type == "cuda" else "cpu"
    
    with torch.no_grad():
        with torch.amp.autocast(device_type=device_type, enabled=trainer.use_amp):
            input_ids = batch["input_ids"].to(next(model.parameters()).device)
            attention_mask = batch["attention_mask"].to(next(model.parameters()).device)
            outputs = model(input_ids, attention_mask)
            
            # Compute loss
            labels_spec = batch["labels_specialist"].to(next(model.parameters()).device)
            labels_sev = batch["labels_severity"].to(next(model.parameters()).device)
            loss_fn = JointLoss()
            loss_dict = model.compute_loss(
                outputs.specialist_logits,
                outputs.severity_logits,
                labels_spec,
                labels_sev,
                loss_fn
            )
            trace["Loss Computation"] = {
                "input_dtypes": [str(outputs.specialist_logits.dtype), str(outputs.severity_logits.dtype)],
                "output_dtypes": [str(loss_dict["joint_loss"].dtype)]
            }
            
    # Remove hooks
    for h in hooks:
        h.remove()
        
    # Check DType Contract
    contract_results = {}
    def check_rule(name, key, expected, actual_list):
        if not actual_list:
            contract_results[f"{name}_{key}"] = f"Fail: Output empty"
            return False
        passed = all(expected in a for a in actual_list)
        contract_results[f"{name}_{key}"] = "Pass" if passed else f"Fail: expected {expected}, got {actual_list}"
        return passed

    all_passed = True
    
    if trainer.use_amp:
        all_passed &= check_rule("Encoder", "outputs", "float16", trace["Encoder"]["output_dtypes"])
    else:
        all_passed &= check_rule("Encoder", "outputs", "float32", trace["Encoder"]["output_dtypes"])
        
    all_passed &= check_rule("Evidence Synthesizer", "inputs", "float32", trace["Evidence Synthesizer"]["input_dtypes"])
    all_passed &= check_rule("Evidence Synthesizer", "outputs", "float32", trace["Evidence Synthesizer"]["output_dtypes"])
    all_passed &= check_rule("Router", "inputs", "float32", trace["Router"]["input_dtypes"])
    all_passed &= check_rule("Router", "outputs", "float32", trace["Router"]["output_dtypes"])
    all_passed &= check_rule("Execution Engine", "inputs", "float32", trace["Execution Engine"]["input_dtypes"])
    all_passed &= check_rule("Execution Engine", "outputs", "float32", trace["Execution Engine"]["output_dtypes"])
    
    for idx in range(len(model.blocks)):
        block_name = f"Thought Block {idx+1}"
        if block_name in trace:
            all_passed &= check_rule(block_name, "inputs", "float32", trace[block_name]["input_dtypes"])
            all_passed &= check_rule(block_name, "outputs", "float32", trace[block_name]["output_dtypes"])
        else:
            contract_results[f"{block_name}_inputs"] = "Skipped (Not Routed)"
            contract_results[f"{block_name}_outputs"] = "Skipped (Not Routed)"
    all_passed &= check_rule("Consistency Projection", "inputs", "float32", trace["Consistency Projection"]["input_dtypes"])
    all_passed &= check_rule("Consistency Projection", "outputs", "float32", trace["Consistency Projection"]["output_dtypes"])
    all_passed &= check_rule("Prediction Heads (Specialist)", "inputs", "float32", trace["Prediction Heads (Specialist)"]["input_dtypes"])
    
    # Save Report
    report = {
        "contract_checks": contract_results,
        "full_trace": trace,
        "amp_enabled": trainer.use_amp,
        "device": str(next(model.parameters()).device),
        "validation_passed": all_passed
    }
    
    report_path = dirs["exports"] / "dtype_report.json"
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=4)
        
    print(f"Contract Checks Status: {'SUCCESS' if all_passed else 'FAILED'}")
    print(json.dumps(contract_results, indent=4))
    print(f"Exported dtype_report.json to {report_path}")
    
    if not all_passed:
        raise ValueError(f"DType Contract Violation detected: {contract_results}")

run_dtype_trace_and_validate()


### 10.2. Single-Step Training Dry Run

In [ ]:
# ==========================================
# SECTION 10.2: TRAINING DRY RUN
# ==========================================
import json
import torch
from src.model import JointLoss

def run_training_dry_run():
    print("="*60)
    print("RUNNING TRAINING DRY RUN (ONE OPTIMIZATION STEP)")
    print("="*60)
    
    if 'model' not in globals() or 'trainer' not in globals() or 'train_loader' not in globals():
        print("Reconstructing runtime components...")
        config, dirs, tokenizer, validator, train_loader, val_loader, test_loader, model, trainer = recover_runtime()
    else:
        model = globals()['model']
        trainer = globals()['trainer']
        train_loader = globals()['train_loader']
        dirs = globals()['dirs']
        
    batch = next(iter(train_loader))
    model.train()
    trainer.optimizer.zero_grad()
    
    device = trainer.device
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels_spec = batch["labels_specialist"].to(device)
    labels_sev = batch["labels_severity"].to(device)
    
    device_type = "cuda" if device.type == "cuda" else "cpu"
    
    try:
        with torch.amp.autocast(device_type=device_type, enabled=trainer.use_amp):
            outputs = model(input_ids, attention_mask)
            
            from models.emergent_path_triage.hooks import apply_loss_hook
            loss_fn = JointLoss()
            loss_dict = apply_loss_hook(
                model,
                outputs.specialist_logits,
                outputs.severity_logits,
                labels_spec,
                labels_sev,
                loss_fn
            )
            loss = loss_dict["joint_loss"]
            
        trainer.scaler.scale(loss).backward()
        trainer.scaler.step(trainer.optimizer)
        trainer.scaler.update()
        
        if trainer.scheduler is not None:
            trainer.scheduler.step()
            
        validation_status = {
            "status": "Success",
            "loss_value": float(loss.item()),
            "device": str(device),
            "amp_enabled": trainer.use_amp,
            "scaler_scale": float(trainer.scaler.get_scale()) if hasattr(trainer.scaler, 'get_scale') else 1.0
        }
        print("Training dry run completed successfully!")
    except Exception as e:
        validation_status = {
            "status": "Failed",
            "error": str(e)
        }
        print(f"Training dry run failed: {e}")
        raise e
        
    report_path = dirs["exports"] / "runtime_validation.json"
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(validation_status, f, indent=4)
    print(f"Exported runtime_validation.json to {report_path}")

run_training_dry_run()


### 10.3. Session Recovery & Evaluation Dry Run

In [ ]:
# ==========================================
# SECTION 10.3: RECOVERY DRY RUN
# ==========================================
import json
import torch
from src.model import JointLoss

def run_recovery_dry_run():
    print("="*60)
    print("RUNNING RECOVERY DRY RUN (SIMULATING FRESH NAMESPACE)")
    print("="*60)
    
    config, dirs, tokenizer, validator, train_loader, val_loader, test_loader, model, trainer = recover_runtime()
    
    best_ckpt = dirs["checkpoints"] / "best_model.pt"
    if not best_ckpt.exists():
        best_ckpt = dirs["experiments"] / "best_model.pt"
        
    print(f"Simulating evaluation recovery from: {best_ckpt}")
    trainer.load_checkpoint(best_ckpt)
    
    model.eval()
    batch = next(iter(test_loader))
    
    device = trainer.device
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels_spec = batch["labels_specialist"].to(device)
    labels_sev = batch["labels_severity"].to(device)
    
    device_type = "cuda" if device.type == "cuda" else "cpu"
    
    try:
        with torch.no_grad():
            with torch.amp.autocast(device_type=device_type, enabled=trainer.use_amp):
                outputs = model(input_ids, attention_mask)
                loss_fn = JointLoss()
                loss_dict = model.compute_loss(
                    outputs.specialist_logits,
                    outputs.severity_logits,
                    labels_spec,
                    labels_sev,
                    loss_fn
                )
                
        eval_status = {
            "status": "Success",
            "eval_loss": float(loss_dict["joint_loss"].item()),
            "device": str(device),
            "amp_enabled": trainer.use_amp
        }
        print("Evaluation recovery step run completed successfully!")
    except Exception as e:
        eval_status = {
            "status": "Failed",
            "error": str(e)
        }
        print(f"Evaluation recovery dry run failed: {e}")
        raise e
        
    checkpoint_val_path = dirs["exports"] / "checkpoint_validation.json"
    recovery_val_path = dirs["exports"] / "recovery_validation.json"
    health_report_path = dirs["exports"] / "health_report.json"
    
    with open(checkpoint_val_path, "w", encoding="utf-8") as f:
        json.dump(eval_status, f, indent=4)
        
    with open(recovery_val_path, "w", encoding="utf-8") as f:
        json.dump({"recovery_status": "Success", "checkpoint_used": str(best_ckpt)}, f, indent=4)
        
    health_report = {
        "cuda_available": torch.cuda.is_available(),
        "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "pytorch_version": torch.__version__,
        "amp_supported": trainer.env_meta["mixed_precision_available"],
        "validation_passed": True
    }
    with open(health_report_path, "w", encoding="utf-8") as f:
        json.dump(health_report, f, indent=4)
        
    print(f"Exported checkpoint_validation.json to {checkpoint_val_path}")
    print(f"Exported recovery_validation.json to {recovery_val_path}")
    print(f"Exported health_report.json to {health_report_path}")

run_recovery_dry_run()


In [37]:
!find /content/MediTriageAI/models -name "__init__.py"

/content/MediTriageAI/models/__init__.py
/content/MediTriageAI/models/emergent_path_triage/__init__.py


In [40]:
!pwd
!ls -la

/content/MediTriageAI
total 328
drwxr-xr-x 12 root root  4096 Jul 21 09:36 .
drwxr-xr-x  1 root root  4096 Jul 21 09:36 ..
drwxr-xr-x  3 root root  4096 Jul 21 09:36 analysis
-rw-r--r--  1 root root 26555 Jul 21 09:36 analysis_report.html
-rw-r--r--  1 root root 18412 Jul 21 09:36 analysis_report.md
-rw-r--r--  1 root root  4629 Jul 21 09:36 ANNOTATION_INTEGRITY_CHECK.md
-rw-r--r--  1 root root  8117 Jul 21 09:36 BASELINE_RESULTS.md
-rw-r--r--  1 root root  2461 Jul 21 09:36 BUGFIX_VERIFICATION.md
-rw-r--r--  1 root root   319 Jul 21 09:36 CITATION.cff
-rw-r--r--  1 root root  1685 Jul 21 09:36 CLASS_DISTRIBUTION.md
-rw-r--r--  1 root root    38 Jul 21 09:36 clinician_overlap_stats.json
-rw-r--r--  1 root root   151 Jul 21 09:36 CLINICIAN_TEST_SET_V3.md
-rw-r--r--  1 root root  2219 Jul 21 09:36 COLAB_SETUP.md
-rw-r--r--  1 root root  5778 Jul 21 09:36 CONFUSION_ANALYSIS.md
drwxr-xr-x  5 root root  4096 Jul 21 09:36 dashboard_web
-rw-r--r--  1 root root  6760 Jul 21 09:36 DEMO_SCRIPT.m

In [41]:
!find /content/MediTriageAI -maxdepth 3 -type d

/content/MediTriageAI
/content/MediTriageAI/meditriage
/content/MediTriageAI/meditriage/data
/content/MediTriageAI/meditriage/data/processed
/content/MediTriageAI/meditriage/data/raw
/content/MediTriageAI/.git
/content/MediTriageAI/.git/info
/content/MediTriageAI/.git/refs
/content/MediTriageAI/.git/refs/heads
/content/MediTriageAI/.git/refs/tags
/content/MediTriageAI/.git/refs/remotes
/content/MediTriageAI/.git/logs
/content/MediTriageAI/.git/logs/refs
/content/MediTriageAI/.git/hooks
/content/MediTriageAI/.git/objects
/content/MediTriageAI/.git/objects/info
/content/MediTriageAI/.git/objects/pack
/content/MediTriageAI/.git/branches
/content/MediTriageAI/dashboard_web
/content/MediTriageAI/dashboard_web/data
/content/MediTriageAI/dashboard_web/css
/content/MediTriageAI/dashboard_web/js
/content/MediTriageAI/src
/content/MediTriageAI/analysis
/content/MediTriageAI/analysis/results
/content/MediTriageAI/analysis/results/experiment_2026_07_20_070026
/content/MediTriageAI/analysis/results

In [46]:
!grep -n "def load_checkpoint" /content/MediTriageAI/src/trainer.py

401:    def load_checkpoint(self, path: Path) -> int:


In [47]:
!sed -n '250,420p' /content/MediTriageAI/src/trainer.py

                self.optimizer.zero_grad()
                if self.scheduler is not None:
                    self.scheduler.step()

            # Record metrics
            spec_preds = outputs.specialist_logits.argmax(dim=-1)
            sev_preds = outputs.severity_logits.argmax(dim=-1)
            tracker.update(loss_dict, spec_preds, labels_spec, sev_preds, labels_sev)

        metrics = tracker.get_summary()
        metrics["lr"] = self.optimizer.param_groups[-1]["lr"]
        return metrics

    def validate(self) -> dict[str, float]:
        """Perform evaluation pass over validation split."""
        self.model.eval()
        tracker = MetricTracker()

        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels_spec = batch["labels_specialist"].to(self.device)
                labels_sev = batch["labels_severi

In [52]:
!grep -Rn "trainer =" /content/MediTriageAI

/content/MediTriageAI/EPATH_CO_REASON_Training.ipynb:427:    "trainer = EmergentTrainer(\n",
/content/MediTriageAI/scripts/run_baseline.py:120:    trainer = EmergentTrainer(
/content/MediTriageAI/tests/test_emergent_path_triage.py:1612:    trainer = EmergentTrainer(
/content/MediTriageAI/tests/test_emergent_path_triage.py:1631:    new_trainer = EmergentTrainer(
/content/MediTriageAI/tests/test_emergent_path_triage.py:1686:    trainer = EmergentTrainer(


In [53]:
!grep -Rn "test_loader" /content/MediTriageAI

/content/MediTriageAI/EPATH_CO_REASON_Training.ipynb:401:    "test_loader = get_dataloader(create_ds(test_df), batch_size=EXPERIMENT_CONFIG[\"batch_size\"], shuffle=False)\n",
/content/MediTriageAI/EPATH_CO_REASON_Training.ipynb:483:    "    for batch in test_loader:\n",
grep: /content/MediTriageAI/src/__pycache__/trainer.cpython-312.pyc: binary file matches
/content/MediTriageAI/src/trainer.py:124:        test_loader: DataLoader | None = None,
/content/MediTriageAI/src/trainer.py:131:        self.test_loader = test_loader
/content/MediTriageAI/scripts/colab_train.py:62:    test_loader = DataLoader(test_dataset, batch_size=32)
/content/MediTriageAI/scripts/colab_train.py:147:        for batch in test_loader:
/content/MediTriageAI/scripts/run_experiment.py:193:    metrics = evaluator.run_evaluation(artifacts.model, artifacts.tokenizer, artifacts.test_loader, artifacts.config)
/content/MediTriageAI/scripts/train.py:69:    test_loader: DataLoader
/content/MediTriageAI/scripts/train.py:96:

In [54]:
!grep -Rn "model =" /content/MediTriageAI

/content/MediTriageAI/EPATH_CO_REASON_Training.ipynb:406:    "model = model_meta.build(None, triage_config=config)\n",
/content/MediTriageAI/src/trainer.py:127:        self.model = model
/content/MediTriageAI/src/metrics.py:195:    novel_model = next((item for item in ranked if item.get("is_novel_contribution")), ranked[0])
/content/MediTriageAI/analysis/io.py:70:    built_model = model_instance.build(None)
/content/MediTriageAI/scripts/colab_train.py:66:    model = DistilBertClassifier(len(label_list)).to(device)
/content/MediTriageAI/scripts/train.py:88:    built_model = model_meta.build(None)
/content/MediTriageAI/scripts/serve_dashboard.py:108:            model = data.get("model", "xlm_roberta")
/content/MediTriageAI/scripts/diagnose_baseline.py:86:    model = model_meta.build(TinyConfig(), triage_config=config)
/content/MediTriageAI/scripts/run_baseline.py:112:    model = model_meta.build(TinyConfig(), triage_config=config)
/content/MediTriageAI/scripts/infer.py:89:    model = MOD

In [50]:
for var in [
    "trainer",
    "model",
    "test_loader",
    "validator",
    "dirs",
    "exp_base",
]:
    print(f"{var}: {'YES' if var in globals() else 'NO'}")

trainer: NO
model: NO
test_loader: NO
validator: YES
dirs: YES
exp_base: YES
